# MACE + GPU4PySCF (TD-DFT) ハイブリッド計算ノートブック

このノートブックは、機械学習ポテンシャルである **MACE (MACE-OFF23)** を用いてGPU上で超高速に構造最適化（ASE経由）を行い、得られた最安定構造を用いて **GPU4PySCF** でTD-DFT（S1, T1, 振動子強度）およびHOMO/LUMO・双極子モーメントを算出するハイブリッドワークフローを実行します。

In [1]:
# 1. ライブラリの読み込み

# 標準ライブラリ
import os      # 入出力ファイルのパス操作とディレクトリ作成
import re      # 材料名をファイル名に使える文字へ置換
import time    # 分子ごとの計算時間を計測
import io      # 最適化構造をXYZ文字列として受け取る
import gc      # 計算オブジェクトを破棄し、GPUメモリを返せるようにする
import ctypes  # cuTENSORの共有ライブラリを先に読み込む

# 数値・表データ
import numpy as np   # 軌道番号や双極子モーメントなどの数値処理
import pandas as pd  # 入力CSVの読み込みと計算結果の書き出し

# 分子構造（SMILESから3次元座標を作る）
import rdkit                    # バージョン確認用
from rdkit import Chem          # SMILESの解釈、水素付加、形式電荷
from rdkit.Chem import AllChem  # 初期3次元構造の生成と力場による最適化

# 機械学習ポテンシャルによる構造最適化（MACE / ASE）
import torch                     # MACEがGPUで使う計算基盤
import ase                       # 原子構造と最適化ループの土台。バージョン確認用
import mace                      # MACE-OFF23本体。バージョン確認用
from ase import Atoms            # RDKitの座標をASEの原子オブジェクトへ渡す
from ase.optimize import LBFGS   # MACEの力で構造を最適化する
from ase.io import write         # 最適化構造をXYZファイルへ書く
from mace.calculators import mace_off  # 有機分子向けMACE-OFF23の力場

# GPU計算の土台
import cupy      # GPU上の配列演算。gpu4pyscfが内部で使う
import cutensor  # NVIDIA cuTENSOR本体。共有ライブラリの場所を得る
cutensor_dir = os.path.join(list(cutensor.__path__)[0], "lib")
ctypes.CDLL(os.path.join(cutensor_dir, "libcutensor.so.2"), mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(os.path.join(cutensor_dir, "libcutensorMg.so.2"), mode=ctypes.RTLD_GLOBAL)
from cupy_backends.cuda.libs import cutensor as cutensor_lib  # CuPyから使うcuTENSOR

# 量子化学計算（PySCF / GPU4PySCF）
import pyscf       # 量子化学計算の本体
import gpu4pyscf   # PySCFのGPU版。DFTとTD-DFTをGPUで実行する
from pyscf import gto  # 座標・基底関数・電荷から計算用の分子を定義
from pyscf.tools import cubegen  # HOMO/LUMOをcubeファイルとして書き出す
from gpu4pyscf.dft import rks as dft_rks    # GPU上のDFT（基底状態のエネルギーと軌道）
from gpu4pyscf.tdscf import rks as td_rks  # GPU上のTD-DFT（励起状態 S1/T1）

print(f"RDKit バージョン: {rdkit.__version__}")
print(f"PySCF バージョン: {pyscf.__version__}")
print(f"GPU4PySCF バージョン: {gpu4pyscf.__version__}")
print(f"CuPy バージョン: {cupy.__version__}")
print(f"cuTENSOR バージョン: {cutensor_lib.get_version()}")
print(f"PyTorch バージョン: {torch.__version__} (CUDA利用可能: {torch.cuda.is_available()})")
print(f"ASE バージョン: {ase.__version__}")
print(f"MACE バージョン: {mace.__version__}")

def release_gpu():
    # CuPyのプールに残ったVRAMと、MACEが使ったPyTorchのキャッシュを返す
    gc.collect()
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = torch

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
RDKit バージョン: 2026.03.3
PySCF バージョン: 2.13.1
GPU4PySCF バージョン: 1.7.6
CuPy バージョン: 14.1.1
cuTENSOR バージョン: 20700
PyTorch バージョン: 2.5.1+cu121 (CUDA利用可能: True)
ASE バージョン: 3.29.0
MACE バージョン: 0.3.16


In [2]:
# 2. データセットの読み込み
csv_path = 'data/test_dft.csv'                       # SMILESを持つ入力
output_csv = 'output_mace/test_mace_results.csv'     # 結果の保存先。存在すればここから再開する

df = pd.read_csv(csv_path)
print(f"読み込まれた化合物数: {len(df)}")
df.head()

読み込まれた化合物数: 2


,material,SMILES
0,ICzCN,C=1C(N2C3=CC=CC=C3C4=CC=C5N(C6=CC=CC=C6C5=C24)...
1,ICzCYP,C1=2C=3C(=CC=CC3)N(C1=C4C=5C(N(C=6C=CC=CC6)C4=...


In [3]:
# 3. SMILESからASE Atomsオブジェクトを作る
# DFT側の smiles_to_3d_block と同じ流れで、戻り値だけASE用に変えている
def smiles_to_ase_atoms(smiles):
    """
    SMILESをRDKitで3D構造化し、ASEのAtomsに変換する。
    戻り値: (Atoms, 電荷)。失敗時は (None, None)。
    """
    # SMILESを分子オブジェクトにする
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None

    # 水素を足し、初期3D座標を作る
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    if AllChem.EmbedMolecule(mol, params) == -1:
        # 決め打ちの埋め込みが失敗したら、ランダム初期座標で再試行する
        params.useRandomCoords = True
        if AllChem.EmbedMolecule(mol, params) == -1:
            return None, None

    # 力場で大まかな形を整える。MMFFが組めないときは例外ではなく -1。そのときだけUFFへ替える
    try:
        mmff_failed = AllChem.MMFFOptimizeMolecule(mol) == -1
    except Exception:
        mmff_failed = True
    if mmff_failed:
        try:
            AllChem.UFFOptimizeMolecule(mol)
        except Exception:
            pass

    # 原子種と座標をASEへ渡す
    conf = mol.GetConformer()
    positions = []
    symbols = []
    for i, atom in enumerate(mol.GetAtoms()):
        pos = conf.GetAtomPosition(i)
        symbols.append(atom.GetSymbol())
        positions.append([pos.x, pos.y, pos.z])

    atoms = Atoms(symbols=symbols, positions=positions)
    charge = Chem.GetFormalCharge(mol)
    return atoms, charge


In [4]:
# 4. MACEで構造最適化し、GPU4PySCFで電子状態とTD-DFTを計算する
def run_mace_opt_and_tddft(atoms, charge=0, xc='b3lyp', basis='6-31g', fmax=0.05, mol_name='unknown', out_dir='output_mace'):
    """
    MACE-OFF23で構造を最適化し、その座標でGPU4PySCFのDFTとTD-DFTを行う。
    最適化構造のXYZ、HOMO/LUMOのcube、PySCFのlogを書く。

    dft_sandbox との違いは構造最適化だけである。
    こちらはDFTの力ではなくMACEの力を使い、Moldenとチェックポイントは出さない。
    計算時間はここでは測らず、バッチ処理側で測る。

    流れは次のとおり。
    1. MACE-OFF23で構造最適化し、xyzを保存する
    2. 最適化構造でSCFを1回行う（ここが電子状態の本番計算）
    3. HOMO/LUMO、cube、基底状態の双極子、TD-DFT（S1/T1/f）を出す
    """
    # 閉殻RKSでは奇数電子を扱えない。MACEを回す前に落とす
    n_elec = int(sum(atoms.get_atomic_numbers())) - int(charge)
    if n_elec % 2 != 0:
        raise ValueError(
            f"電子数が奇数 ({n_elec}) です。このノートブックは閉殻のRKS専用です。"
            "イオンやラジカルは dft_ip_reorg.ipynb / dft_ea_reorg.ipynb を使ってください。"
        )

    # --- 構造最適化（MACE）。エネルギーはまだDFTではない ---
    print("  -> MACE (機械学習ポテンシャル) による高速構造最適化を開始します...")
    macenet = mace_off(model="medium", device='cuda')  # MACE-OFF23 medium をGPUで使う
    atoms.calc = macenet
    opt = LBFGS(atoms, logfile=None)  # ASEのLBFGS。ログは標準出力へ出さない
    # 既定のステップ上限は事実上無い。収束しなければ失敗にして、次の分子へ進む
    converged = opt.run(fmax=fmax, steps=300)
    atoms.calc = None
    del opt, macenet
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if converged is False:
        raise RuntimeError(f"MACEの構造最適化が300ステップ以内に収束しませんでした (fmax={fmax} eV/Å)。")

    # 最適化座標をxyzへ残し、PySCFへ渡す文字列にもする
    xyz_path = os.path.join(out_dir, f"{mol_name}_opt.xyz")
    write(xyz_path, atoms, format='xyz')
    opt_positions = atoms.get_positions()
    opt_symbols = atoms.get_chemical_symbols()
    coord_list = [f"{sym} {pos[0]} {pos[1]} {pos[2]}" for sym, pos in zip(opt_symbols, opt_positions)]
    opt_coord_string = "; ".join(coord_list)

    # --- 電子状態（GPU4PySCF）。構造はもう動かさない ---
    print("  -> 最適化構造で電子状態計算 (GPU4PySCF) を実行中...")
    log_file = os.path.join(out_dir, f"{mol_name}.log")
    # verbose=4 は反復ログを詳しく残す指定、output はそのログの保存先
    mol = gto.M(atom=opt_coord_string, basis=basis, charge=charge, verbose=4, output=log_file)
    # JもKもRI近似。スクリーニング向けに積分グリッドは level 1
    # dft_sandbox の初期値は level 3（PySCF標準）なので、ここだけ粗い
    mf_opt = dft_rks.RKS(mol, xc=xc).density_fit()
    mf_opt.grids.level = 1
    mf_opt.kernel()

    # 軌道エネルギーからHOMO/LUMOを拾う。GPU配列は .get() でNumPyへ戻す
    HARTREE_TO_EV = 27.211386
    mo_energy = mf_opt.mo_energy.get()
    mo_occ = mf_opt.mo_occ.get()
    occupied_indices = np.where(mo_occ > 0)[0]
    unoccupied_indices = np.where(mo_occ == 0)[0]
    if len(occupied_indices) == 0 or len(unoccupied_indices) == 0:
        raise ValueError("分子軌道を正しく識別できませんでした。")
    homo_idx = np.max(occupied_indices)
    lumo_idx = np.min(unoccupied_indices)
    homo_ev = mo_energy[homo_idx] * HARTREE_TO_EV
    lumo_ev = mo_energy[lumo_idx] * HARTREE_TO_EV

    # HOMO/LUMOの軌道図。cubegenはCPU側の係数を受け取る
    print("  -> HOMO / LUMO のCubeファイルを生成中...")
    mo_coeff_cpu = mf_opt.mo_coeff.get()
    cubegen.orbital(mol, os.path.join(out_dir, f"{mol_name}_HOMO.cube"), mo_coeff_cpu[:, homo_idx])
    cubegen.orbital(mol, os.path.join(out_dir, f"{mol_name}_LUMO.cube"), mo_coeff_cpu[:, lumo_idx])

    # 基底状態の双極子モーメントの大きさ（励起状態ではない）
    print("  -> 双極子モーメントを計算中...")
    dip_mom = mf_opt.dip_moment(unit='Debye')
    dipole_val = np.linalg.norm(dip_mom)

    # --- TD-DFT。一重項S1と三重項T1 ---
    print("  -> TD-DFTによる一重項/三重項の計算を実行中 (S1, T1)...")
    td_s = td_rks.TDDFT(mf_opt)
    td_s.nstates = 3
    td_s.kernel()
    s1_ev = td_s.e[0] * HARTREE_TO_EV
    f_s1 = td_s.oscillator_strength()[0]
    # 一重項の中間配列を捨ててから三重項に入る
    del td_s
    release_gpu()

    td_t = td_rks.TDDFT(mf_opt)
    td_t.singlet = False
    td_t.nstates = 3
    td_t.kernel()
    t1_ev = td_t.e[0] * HARTREE_TO_EV

    # 呼び出し側がCSVへ残せるよう、最適化構造の中身も返す
    f_io = io.StringIO()
    write(f_io, atoms, format='xyz')
    opt_xyz = f_io.getvalue()

    e_tot = float(mf_opt.e_tot)
    del mf_opt, td_t, mol
    release_gpu()
    return homo_ev, lumo_ev, s1_ev, t1_ev, f_s1, dipole_val, e_tot, opt_xyz


In [5]:
# 5. 計算条件とバッチ処理
# 汎関数と基底関数。dft_sandbox と同じ指定名を使う
XC_FUNCTIONAL = 'b3lyp'

# 主な基底関数の指定名:
# 'sto-3g' '6-31g' '6-31g*' '6-31g**' '6-311g**' 'def2-svp'
BASIS_SET = '6-31g*'

# MACE最適化の打ち切り。原子にかかる力の最大値がこの値（eV/Å）未満なら収束とする
FMAX = 0.05

# 結果CSVがあればそこから再開し、なければ入力CSVから表を作る
if os.path.exists(output_csv):
    df_results = pd.read_csv(output_csv)
    print(f"既存の結果CSV（{output_csv}）を読み込んで計算を再開します。")
else:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"入力用CSVファイルが見つかりません: {csv_path}")
    df_results = pd.read_csv(csv_path)
    print(f"オリジナルのデータセット（{csv_path}）を読み込みました。")

def resolve_input_columns(df):
    # READMEどおり、材料IDは material / index、構造は SMILES という列名で拾う
    # 列名が無いCSVだけ、先頭2列にフォールバックする
    lookup = {}
    for col in df.columns:
        key = str(col).strip().lstrip("\ufeff").lower()
        lookup.setdefault(key, col)
    smiles_col = lookup.get("smiles")
    id_col = None
    for key in ("material", "index"):
        if key in lookup and lookup[key] != smiles_col:
            id_col = lookup[key]
            break
    if smiles_col is not None and id_col is not None:
        print(f"入力データ設定: '{id_col}' を材料ID、'{smiles_col}' をSMILESとして認識しました。")
        return id_col, smiles_col
    if len(df.columns) >= 2:
        id_col = df.columns[0]
        smiles_col = df.columns[1]
        print(
            "列名 material/index と SMILES が見つからないため、"
            f"1列目 '{id_col}' を材料ID、2列目 '{smiles_col}' をSMILESとして使います。"
        )
        return id_col, smiles_col
    raise ValueError("CSVには材料ID（material または index）と SMILES の2列が必要です。")

# 列名で材料IDとSMILESを拾う。順番が逆でも、間に別の列があってもよい
id_col, smiles_col = resolve_input_columns(df_results)
df_results.rename(columns={id_col: 'Material', smiles_col: 'SMILES'}, inplace=True)

# 結果列が無ければ追加する。elapsed_sec は成功・失敗のどちらでも秒数を残す
target_cols = {
    'HOMO_eV': np.nan,
    'LUMO_eV': np.nan,
    'S1_eV': np.nan,
    'T1_eV': np.nan,
    'delta_Est_eV': np.nan,
    'f': np.nan,
    'dipole_Debye': np.nan,
    'Total_Energy_Hartree': np.nan,
    'elapsed_sec': np.nan,
    'state': 'pending'
}
for col, default_val in target_cols.items():
    if col not in df_results.columns:
        df_results[col] = default_val

OUT_DIR = 'output_mace'
os.makedirs(OUT_DIR, exist_ok=True)
print("=== バッチ計算を開始します ===")
for index, row in df_results.iterrows():
    name = row['Material']
    smiles = row['SMILES']
    safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', str(name))
    state = row['state']

    # completed と fail は再計算しない
    if state in ['completed', 'fail']:
        print(f"\n[{index + 1}/{len(df_results)}] スキップします: {name} (SMILES: {smiles}) - すでにステータスが {state} です")
        continue

    print(f"\n[{index + 1}/{len(df_results)}] 計算中: {name} (SMILES: {smiles})")
    df_results.at[index, 'state'] = 'running'
    df_results.to_csv(output_csv, index=False)

    mol_out_dir = os.path.join(OUT_DIR, safe_name)
    os.makedirs(mol_out_dir, exist_ok=True)

    # ステップ 1: SMILESからASEの原子構造を作る。ここは計算時間に含めない
    try:
        atoms, charge = smiles_to_ase_atoms(smiles)
        if atoms is None:
            raise ValueError("RDKitによる3D構造生成に失敗しました。")
    except Exception as e:
        print(f"  [エラー] 3D構造の生成に失敗しました: {e}")
        df_results.at[index, 'state'] = 'fail'
        df_results.to_csv(output_csv, index=False)
        continue

    # ステップ 2: MACE最適化とTD-DFT。成功しても失敗しても経過秒を残す
    t0 = time.time()
    try:
        homo, lumo, s1, t1, f_s1, dipole, energy, opt_xyz = run_mace_opt_and_tddft(
            atoms, charge=charge, xc=XC_FUNCTIONAL, basis=BASIS_SET, fmax=FMAX, mol_name=safe_name, out_dir=mol_out_dir
        )
        elapsed = time.time() - t0
        delta_est = s1 - t1
        print(f"  [成功] 計算完了 ({elapsed:.1f}秒). S1: {s1:.3f} eV, T1: {t1:.3f} eV, ΔEst: {delta_est:.3f} eV, f: {f_s1:.4f}, dipole: {dipole:.3f} D")

        df_results.at[index, 'HOMO_eV'] = homo
        df_results.at[index, 'LUMO_eV'] = lumo
        df_results.at[index, 'S1_eV'] = s1
        df_results.at[index, 'T1_eV'] = t1
        df_results.at[index, 'delta_Est_eV'] = delta_est
        df_results.at[index, 'f'] = f_s1
        df_results.at[index, 'dipole_Debye'] = dipole
        df_results.at[index, 'Total_Energy_Hartree'] = energy
        df_results.at[index, 'elapsed_sec'] = elapsed
        df_results.at[index, 'state'] = 'completed'
    except Exception as e:
        elapsed = time.time() - t0
        print(f"  [エラー] 計算が異常終了しました ({elapsed:.1f}秒): {e}")
        df_results.at[index, 'elapsed_sec'] = elapsed
        df_results.at[index, 'state'] = 'fail'

    # 成功しても失敗しても、次の分子に入る前にVRAMを返す
    release_gpu()

    # 1分子終わるごとにCSVへ書く。途中で止まってもそこまでは残る
    df_results.to_csv(output_csv, index=False)
    print(f"  結果を一時保存しました: {output_csv}")


オリジナルのデータセット（data/test_dft.csv）を読み込みました。
入力データ設定: 1列目 'material' を材料ID、2列目 'SMILES' をSMILESとして認識しました。
=== バッチ計算を開始します ===

[1/2] 計算中: ICzCN (SMILES: C=1C(N2C3=CC=CC=C3C4=CC=C5N(C6=CC=CC=C6C5=C24)C7=CC=CC=C7)=CC=C(C1)C8=C(C(C=9C=CC(N%10C=%11C(C%12=CC=C%13N(C=%14C(C%13=C%10%12)=CC=CC%14)C=%15C=CC=CC%15)=CC=CC%11)=CC9)=CN=C8)C#N)
  -> MACE (機械学習ポテンシャル) による高速構造最適化を開始します...
Using MACE-OFF23 MODEL for MACECalculator with /home/kawano/.cache/mace/MACE-OFF23_medium.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/mace/calculators/mace.py:226: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=model_path, map_

  -> 最適化構造で電子状態計算 (GPU4PySCF) を実行中...
output file: output_mace/ICzCN/ICzCN.log
  -> HOMO / LUMO のCubeファイルを生成中...
  -> 双極子モーメントを計算中...
  -> TD-DFTによる一重項/三重項の計算を実行中 (S1, T1)...
  [成功] 計算完了 (10481.1秒). S1: 2.811 eV, T1: 2.804 eV, ΔEst: 0.008 eV, f: 0.0037, dipole: 2.679 D
  結果を一時保存しました: output_mace/test_mace_results.csv

[2/2] 計算中: ICzCYP (SMILES: C1=2C=3C(=CC=CC3)N(C1=C4C=5C(N(C=6C=CC=CC6)C4=CC2)=CC=CC5)C=7C=CC(=CC7)C8=C(C=CC(=N8)C=9C=CC(N%10C=%11C(=CC=CC%11)C=%12C%10=C%13C%14=CC=CC=C%14N(C%13=CC%12)C=%15C=CC=CC%15)=CC9)C#N)
  -> MACE (機械学習ポテンシャル) による高速構造最適化を開始します...
Using MACE-OFF23 MODEL for MACECalculator with /home/kawano/.cache/mace/MACE-OFF23_medium.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/mace/calculators/mace.py:226: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=model_path, map_

  -> 最適化構造で電子状態計算 (GPU4PySCF) を実行中...
output file: output_mace/ICzCYP/ICzCYP.log
  -> HOMO / LUMO のCubeファイルを生成中...
  -> 双極子モーメントを計算中...
  -> TD-DFTによる一重項/三重項の計算を実行中 (S1, T1)...
  [成功] 計算完了 (11066.8秒). S1: 2.594 eV, T1: 2.589 eV, ΔEst: 0.004 eV, f: 0.0017, dipole: 3.851 D
  結果を一時保存しました: output_mace/test_mace_results.csv


In [6]:
# 6. 結果の確認
# elapsed_sec は MACE最適化からTD-DFTまでの秒数。3D構造の生成は含まない
print("バッチ処理が完了しました。最終結果の概要は以下の通りです:")
df_results.head()

バッチ処理が完了しました。最終結果の概要は以下の通りです:


,Material,SMILES,HOMO_eV,LUMO_eV,S1_eV,T1_eV,delta_Est_eV,f,dipole_Debye,Total_Energy_Hartree,elapsed_sec,state
0,ICzCN,C=1C(N2C3=CC=CC=C3C4=CC=C5N(C6=CC=CC=C6C5=C24)...,-5.079473,-1.924296,2.811330,2.803535,0.007795,0.003703,2.678994,-2867.635758,10481.072753,completed
1,ICzCYP,C1=2C=3C(=CC=CC3)N(C1=C4C=5C(N(C=6C=CC=CC6)C4=...,-4.961787,-2.038282,2.593716,2.589404,0.004312,0.001660,3.851199,-2867.647472,11066.795167,completed


In [7]:
# 7. failした材料の抽出と保存
# バッチが途中で止まっても、保存済みCSVから失敗分だけ切り出せる
if os.path.exists(output_csv):
    df_final = pd.read_csv(output_csv)
    df_fail = df_final[df_final['state'] == 'fail']
    
    if len(df_fail) > 0:
        base_name, ext = os.path.splitext(output_csv)
        fail_csv = f"{base_name}_fail{ext}"
        df_fail.to_csv(fail_csv, index=False)
        print(f"failした材料（{len(df_fail)}件）を抽出して保存しました: {fail_csv}")
    else:
        print("failした材料はありませんでした。おめでとうございます！")
else:
    print(f"{output_csv} が見つかりません。")

failした材料はありませんでした。おめでとうございます！
